<a href="https://colab.research.google.com/github/dave-heslop74/EMSC2010-W8-P1/blob/main/EMSC2010_W8_P1_NB1_uXXXXXXX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EMSC2010-W8-P1-NB1
In this notebook we'll again use the river grain size data to fit a first-order (straight-line) polynomial regression. We'll use the ```bambi``` package to set up the regression problem within ```PyMC```.  We'll sample the posterior probability distributions and see how we can make predictions based on the regression.

```bambi``` is not included in Colab, so we first need to install it.

In [ ]:
!pip install bambi #system command to install bambi package

We can now load the packages we'll need into the memory.

In [ ]:
import numpy as np #for working with numerical arrays
import matplotlib.pyplot as plt #for plotting
import bambi as bmb #for automated Bayesian regression
import arviz as az #for analysis of Bayesian models
import pandas as pd #bambi requires us to work with pandas dataframes

Now input the data river gravel data.

In [ ]:
dist = np.array([1, 1.25, 1.75, 2.25, 2.5, 3,  3.25, 3.75, 4.25, 4.5, 4.75, 5, 5.4, 6, 6.35, 7.1, 7.5, 7.9, 8.5, 8.7, 9.1, 9.6, 10.2]) #distance downstream in km
gs = np.array([-10.5411, -10.8376, -10.388, -10.3219, -10.4818, -10.388, -10.1421, -10.0084, -10.0084, -9.9366, -9.5699, -9.7977, -9.6257, -9.388, -9.2527, -9.388, -9.1033, -9.1799, -9.0224, -8.8455, -9.1033, -8.9366, -9.1799]) #grainsize in phi

Plot the data to provide an initial visualization.

In [ ]:
plt.plot(dist,gs,'ok')
plt.xlabel('Distance (km)')
plt.ylabel('Grain size (phi)')
plt.minorticks_on()

Using ```bambi``` we can define the regression problem in a single command.

The ```bambi``` package then automatically creates appropriate priors and samples the posterior distributions.

In [ ]:
data = pd.DataFrame({"x": dist, "y": gs}) #put the data into a dataframe with variable names "x" and "y".
model = bmb.Model("y ~ x", data) #setup the first-order polynomial model
idata = model.fit(draws=2000, tune=2000, chains=8) #sample the posteriors

We can plot the priors used by ```bambi``` (the variable $x$ corresponds to the gradient with respect to $x$).

In [ ]:
model.plot_priors()
plt.tight_layout()
plt.show()

And with ```arviz``` we can plot the sampled posterior distributions.

In [ ]:
az.plot_trace(idata);
plt.tight_layout() #this spaces the plots out so that they don't overlap

#### Making predictions from the regression model

The regression provides us with information to predict gravel size downstream in the river.

For example, if we consider the posterior distributions for the line intercept and gradient, we can find an envelope in which 95% of the regression lines fall with respect to $x$.

In [ ]:
x_range = np.linspace(1, 11, 100) # Predict across a range of x values from 1 to 11
new_data = pd.DataFrame({"x": x_range}) #dataframe with the new x-values
model.predict(idata, data=new_data, kind='response_params') #predict the distribution of regression lines at each x-value

# Make random draws from the posterior of the regression lines
y_mean_draws = idata.posterior["mu"].values

# Reshape to (total_draws, x_points)
y_mean_draws = y_mean_draws.reshape(-1, len(x_range))

# Compute posterior mean and HDI at each x point
posterior_mean = y_mean_draws.mean(axis=0)
hdi = az.hdi(y_mean_draws, hdi_prob=0.95)

# Plot the results
plt.plot(dist, gs, 'ok', label="Data")
plt.plot(x_range, posterior_mean, color="C1", label="Posterior mean")
plt.fill_between(
    x_range,
    hdi[:, 0],
    hdi[:, 1],
    alpha=0.3,
    color="C1",
    label="95% HDI",
    edgecolor = None
)
plt.xlabel('Distance (km)')
plt.ylabel('Grain size (phi)')
plt.legend()
plt.minorticks_on()

The ```95% HDI``` band represents the 95% probability envelope for the mean gravel size at a given downstream position in the river.

If we also consider the ```sigma``` posterior distribution that represents deviations in grain size away from the regression line, we can make a prediction for future observations. For example, how large will a newly collected piece of gravel be?  

In [ ]:
# Predict across a range of x values
x_range = np.linspace(1, 11, 100) # Predict across a range of x values from 1 to 11
new_data = pd.DataFrame({"x": x_range}) #dataframe with the new x-values
model.predict(idata, data=new_data, kind='response') #predict the distribution of gravel size at each x-value

# Make random draws from the posterior of the gravel size
y_pps_draws = idata.posterior_predictive["y"].values

# Reshape to (total_draws, x_points)
y_pps_draws = y_pps_draws.reshape(-1, len(x_range))

# Compute mean and HDI of the posterior predictive distribution
posterior_mean = y_pps_draws.mean(axis=0)
hdi_mean = az.hdi(idata.posterior["mu"].values.reshape(-1, len(x_range)), hdi_prob=0.95)
hdi_pps = az.hdi(y_pps_draws, hdi_prob=0.95)

# Plot both posterior for regression lines and the observations
plt.plot(dist, gs, 'ok', label="Data")
plt.plot(x_range, posterior_mean, color="C1", label="Posterior mean")
plt.fill_between(
    x_range,
    hdi[:, 0],
    hdi[:, 1],
    alpha=0.3,
    color="C1",
    label="95% HDI",
    edgecolor = None
)

plt.fill_between(
    x_range,
    hdi_pps[:, 0],
    hdi_pps[:, 1],
    alpha=0.2,
    color="C1",
    label="95% HDI (predictive)",
    edgecolor = None
)

plt.legend()
plt.minorticks_on()
plt.xlabel('Distance (km)')
plt.ylabel('Grain size (phi)')

The ```95% HDI (predictive)``` band represents the 95% probability envelope for the size of a new piece of gravel collected at a given downstream position in the river.